# Step 5c — RAG (Retrieval-Augmented Generation)

🇬🇧 **English** (this notebook)

RAG grounds a response in a document you provide: your text is chunked, embedded, and the most relevant chunks are retrieved and injected into the agent's context automatically. Unlike steps 3-5b, `knowledge_sources` does **not** work with a standalone `Agent.kickoff()` call — retrieval only gets wired up through a `Crew`'s own orchestration. So this one exercise needs a minimal inline `Crew` (still defined right here, no external YAML files) instead of a bare `Agent`.

## Background

> Lewis, P., Perez, E., Piktus, A., Petroni, F., Karpukhin, V., Goyal, N., Küttler, H., Lewis, M., Yih, W., Rocktäschel, T., Riedel, S., & Kiela, D. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

![RAG: query encoder and retriever feed top-k documents into a generator](../assets/rag-lewis2020-fig1.png)
*Figure 1 from Lewis et al. (2020). Reproduced for educational use in this course.*

**One practical detail**: embeddings use a separate model from the chat LLM. This notebook points the embedder at Gemini explicitly — you just need `GEMINI_API_KEY` set, even if `MODEL` is a different provider for chat.

## The exercise

Add a `.txt` file with background on your topic to `knowledge/`, then point `TextFileKnowledgeSource` at it. `knowledge/` is resolved relative to wherever the notebook's kernel is running (this repo's `exercises/en/knowledge/`, by default) — not the repo root. The one-agent, one-task `Crew` below is the minimum needed to activate retrieval — you're not editing any project files, everything is defined in this cell.

Same Researcher role once more, now grounded in a private internal document (`knowledge/ai_act_internal_review.txt`, included in this repo) describing a fictional company's own AI system — something no web search or URL fetch could ever find, because it was never published. This is the case RAG is actually for.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

load_dotenv()

# ── Your knowledge document — a private document, not published anywhere ─────
knowledge_source = TextFileKnowledgeSource(file_paths=["ai_act_internal_review.txt"])

# ── Same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)
agent = Agent(role=role, goal=goal, backstory=backstory, verbose=True)

question = (
    "According to our internal review notes, is our Resume Triage Assistant "
    f"classified as high-risk under {topic}, and what compliance gaps remain?"
)
task = Task(
    description=question,
    expected_output="A direct answer grounded in the internal review notes.",
    agent=agent,
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    process=Process.sequential,
    knowledge_sources=[knowledge_source],
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
    embedder={
        "provider": "google-generativeai",
        "config": {"api_key": os.getenv("GEMINI_API_KEY"), "model_name": "gemini-embedding-001"},
    },
)

result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: According to our internal review notes, is our Resume Triage Assistant classified as high-risk under EU  │
│  AI Act, and what compliance gaps remain?                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To provide an accurate assessment, I have reviewed the internal compliance documentation regarding the         │
│  "Resume Triage Assistant." Below is the complete content of the assessment based on the EU AI Act              │
│  classification framework.                                                                                      │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### Internal Compliance Review: Resume Triage Assistant (Project ID: RTA-2024)                                 │
│                                                                                                                 │
│  #### 1. Classification Status                                                                                  │
│  **Classification: High-Risk (Annex III, Point 4a)**                                                            │
│  The Resume Triage Assistant is formally classified as a **High-Risk AI System** under Article 6(1) of the EU   │
│  AI Act. Specifically, it falls under **Annex III, Section 2(a)**, which categorizes "AI systems intended to    │
│  be used for recruitment or selection of natural persons, notably for placing targeted job advertisements,      │
│  analyzing and filtering job applications, and evaluating candidates" as high-risk.                             │
│                                                                                                                 │
│  The system’s function of automating the screening and ranking of applicant resumes constitutes a direct        │
│  impact on access to employment, necessitating full adherence to the requirements set forth in Chapter 2,       │
│  Title III of the Act.                                                                                          │
│                                                                                                                 │
│  #### 2. Identified Compliance Gaps                                                                             │
│  Based on our current internal review, the following critical gaps remain before the system can be considered   │
│  "EU AI Act Compliant":                                                                                         │
│                                                                                                                 │
│  *   **Gap 1: Quality Management System (QMS) (Article 17):**                                                   │
│      We currently lack a documented, auditable QMS that covers the full lifecycle of the RTA. We have not yet   │
│  implemented formal procedures for post-market monitoring or record-keeping related to system performance       │
│  changes.                                                                                                       │
│  *   **Gap 2: Data Governance and Bias Mitigation (Article 10):**                                               │
│      While the training data has been scrubbed of PII, it has not been audited for representativeness or        │
│  systemic bias. The current review notes indicate that 

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 093ae1d2-c4b9-41e9-9b2e-598af096f815                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/093ae1d2-c4b9-41e9-9b2e-598af096f815?access_code=TRA │
│ CE-5a5354484e                                                                                                   │
│ 🔑 Access Code: TRACE-5a5354484e                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

To provide an accurate assessment, I have reviewed the internal compliance documentation regarding the "Resume Triage Assistant." Below is the complete content of the assessment based on the EU AI Act classification framework.

***

### Internal Compliance Review: Resume Triage Assistant (Project ID: RTA-2024)

#### 1. Classification Status
**Classification: High-Risk (Annex III, Point 4a)**
The Resume Triage Assistant is formally classified as a **High-Risk AI System** under Article 6(1) of the EU AI Act. Specifically, it falls under **Annex III, Section 2(a)**, which categorizes "AI systems intended to be used for recruitment or selection of natural persons, notably for placing targeted job advertisements, analyzing and filtering job applications, and evaluating candidates" as high-risk. 

The system’s function of automating the screening and ranking of applicant resumes constitutes a direct impact on access to employment, necessitating full adherence to the requirements set forth in

HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<HTTPSConnection(host='telemetry.crewai.com', port=4319) at 0x12050cb00>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))
HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<HTTPSConnection(host='telemetry.crewai.com', port=4319) at 0x1205f9430>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))


## Your task

1. Run the cell. Does the agent correctly identify the compliance gaps — information that only exists in `knowledge/ai_act_internal_review.txt`, nowhere on the public web?

2. Compare to 5a/5b: could a web search or a URL fetch ever have answered this question? What kind of information is RAG for that tools/MCP aren't?

3. Now pass an empty list instead (`knowledge_sources=[]`) and rerun. Does the agent still answer correctly, or does it admit it doesn't know? That comparison is the point of RAG.

4. Add your own `.txt` file to `knowledge/` with background on your team's topic, and swap in your own agent identity and question.

5. Fill in the **Step 5c** section of `EVALUATION.md` and your final recommendation.

## Stretch goal

Add a PDF instead of a plain text file:
```python
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
PDFKnowledgeSource(file_paths=["your_document.pdf"])
```
Ask a question whose answer appears on a specific page. Does the agent retrieve it?

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.